## Task 1: Build the vocabulary and document-term matrix

This task involves defining our training and test documents, extracting a unique vocabulary from the training data, and then constructing various document-term matrices: a binary matrix, a Term Frequency (TF) matrix, and a TF-IDF matrix.

In [91]:
# Import necessary libraries
import numpy as np
from collections import Counter
import pandas as pd # For better display of matrices

# 1. Training and test data (bag of words) as provided in the skeleton
train_docs = [
    ["free","free","free","buy","discount","combo","pleasure"],   # d1  S
    ["free","free","free","discount","pleasure","smile","smile","smile"], # d2  S
    ["cat","mouse"],                                              # d3  N
    ["cat","cat","dog","dog","dog","dog"],                        # d4  N
    ["mouse"],                                                    # d5  N
]
train_labels = ["S","S","N","N","N"]

test_docs = [
    ["dog","cat","mouse","cat"],   # d6
    ["free","free","smile"],       # d7
]

print("Training Documents:")
for i, doc in enumerate(train_docs):
    print(f"d{i+1}: {doc} (Class: {train_labels[i]})")

print("\nTest Documents:")
for i, doc in enumerate(test_docs):
    print(f"d{i+6}: {doc}")

Training Documents:
d1: ['free', 'free', 'free', 'buy', 'discount', 'combo', 'pleasure'] (Class: S)
d2: ['free', 'free', 'free', 'discount', 'pleasure', 'smile', 'smile', 'smile'] (Class: S)
d3: ['cat', 'mouse'] (Class: N)
d4: ['cat', 'cat', 'dog', 'dog', 'dog', 'dog'] (Class: N)
d5: ['mouse'] (Class: N)

Test Documents:
d6: ['dog', 'cat', 'mouse', 'cat']
d7: ['free', 'free', 'smile']


### Build the vocabulary

First, we extract the full vocabulary from the training set (d1–d5), ensuring all words are lower-cased and unique. We also create a mapping from words to their indices, which will be useful for creating feature vectors.

In [92]:
# 2. Vocabulary
vocab = sorted(list(set(w for doc in train_docs for w in doc)))
word_to_idx = {w: i for i, w in enumerate(vocab)}

print("Full Vocabulary (V):")
print(vocab)
print("\nWord to Index Mapping:")
print(word_to_idx)

Full Vocabulary (V):
['buy', 'cat', 'combo', 'discount', 'dog', 'free', 'mouse', 'pleasure', 'smile']

Word to Index Mapping:
{'buy': 0, 'cat': 1, 'combo': 2, 'discount': 3, 'dog': 4, 'free': 5, 'mouse': 6, 'pleasure': 7, 'smile': 8}


### Binary Feature Vector Matrix

Next, we construct a binary feature vector for each training document. For a term *t* and a document *d*, the value will be 1 if *t* appears in *d*, and 0 otherwise. This forms the `X_bin` matrix.

In [93]:
# 3. Binary matrix
def to_binary(doc, vocab, word_to_idx):
    v = np.zeros(len(vocab), dtype=int)
    for w in set(doc): # Use set(doc) to ensure only presence matters for binary
        if w in word_to_idx:
            v[word_to_idx[w]] = 1
    return v

X_bin = np.array([to_binary(d, vocab, word_to_idx) for d in train_docs])

print("Binary Feature Vector Matrix (X_bin):")
df_X_bin = pd.DataFrame(X_bin, columns=vocab, index=[f'd{i+1}' for i in range(len(train_docs))])
display(df_X_bin)

Binary Feature Vector Matrix (X_bin):


,buy,cat,combo,discount,dog,free,mouse,pleasure,smile
d1,1,0,1,1,0,1,0,1,0
d2,0,0,0,1,0,1,0,1,1
d3,0,1,0,0,0,0,1,0,0
d4,0,1,0,0,1,0,0,0,0
d5,0,0,0,0,0,0,1,0,0


### TF (Term Frequency) and TF-IDF Matrices

Finally, we build the Term Frequency (TF) matrix, where each entry is the raw count of a term in a document. Then, we calculate the Inverse Document Frequency (IDF) for each term using the formula `IDF(t) = log( N / df(t) )`, where `N` is the total number of training documents and `df(t)` is the document frequency of term `t`. The TF-IDF matrix is the product of the TF and IDF matrices.

In [94]:
# 3. TF and TF-IDF matrices
def to_tf(doc, vocab, word_to_idx):
    v = np.zeros(len(vocab), dtype=float)
    for w in doc:
        if w in word_to_idx:
            v[word_to_idx[w]] += 1
    return v

X_tf  = np.array([to_tf(d, vocab, word_to_idx) for d in train_docs])

print("Term Frequency (TF) Matrix (X_tf):")
df_X_tf = pd.DataFrame(X_tf, columns=vocab, index=[f'd{i+1}' for i in range(len(train_docs))])
display(df_X_tf)

N = len(train_docs) # Total number of training documents
df = (X_bin > 0).sum(axis=0) # Document frequency for each term
# Add 1 to df to avoid division by zero for terms not present in any document (though not expected here with current vocab logic)
idf = np.log2(N / np.maximum(df, 1))

print("\nDocument Frequencies (df):")
df_df = pd.DataFrame([df], columns=vocab, index=['df'])
display(df_df)

print("\nInverse Document Frequencies (IDF):")
df_idf = pd.DataFrame([idf], columns=vocab, index=['idf'])
display(df_idf)

X_tfidf = X_tf * idf

print("\nTF-IDF Matrix (X_tfidf):")
df_X_tfidf = pd.DataFrame(X_tfidf, columns=vocab, index=[f'd{i+1}' for i in range(len(train_docs))])
display(df_X_tfidf)

Term Frequency (TF) Matrix (X_tf):


,buy,cat,combo,discount,dog,free,mouse,pleasure,smile
d1,1.0,0.0,1.0,1.0,0.0,3.0,0.0,1.0,0.0
d2,0.0,0.0,0.0,1.0,0.0,3.0,0.0,1.0,3.0
d3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
d4,0.0,2.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0
d5,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0



Document Frequencies (df):


,buy,cat,combo,discount,dog,free,mouse,pleasure,smile
df,1,2,1,2,1,2,2,2,1



Inverse Document Frequencies (IDF):


,buy,cat,combo,discount,dog,free,mouse,pleasure,smile
idf,2.321928,1.321928,2.321928,1.321928,2.321928,1.321928,1.321928,1.321928,2.321928



TF-IDF Matrix (X_tfidf):


,buy,cat,combo,discount,dog,free,mouse,pleasure,smile
d1,2.321928,0.000000,2.321928,1.321928,0.000000,3.965784,0.000000,1.321928,0.000000
d2,0.000000,0.000000,0.000000,1.321928,0.000000,3.965784,0.000000,1.321928,6.965784
d3,0.000000,1.321928,0.000000,0.000000,0.000000,0.000000,1.321928,0.000000,0.000000
d4,0.000000,2.643856,0.000000,0.000000,9.287712,0.000000,0.000000,0.000000,0.000000
d5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.321928,0.000000,0.000000


## Task 2: Information Gain (IG) Calculation

In this section, we calculate the Information Gain for each term in our vocabulary. This involves computing the entropy of the class variable $H(C)$, and for each term $t$, the conditional entropy $H(C|t)$, leading to the Information Gain $IG(t) = H(C) - H(C|t)$. We'll use base-2 logarithms as specified.

In [95]:
import numpy as np
from collections import Counter
import pandas as pd # For better display of matrices

def entropy(labels):
    c = Counter(labels)
    total = sum(c.values())
    # Avoid log2(0) by checking if n > 0
    return -sum((n/total) * np.log2(n/total) for n in c.values() if n > 0)

def info_gain(term_idx, X_bin, labels):
    Hc = entropy(labels)

    # Doğrudan boolean (True/False) maske oluştur
    present = X_bin[:, term_idx] == 1
    absent = ~present # Tilda (~) not işlemini yapar, True'yu False'a çevirir

    # NumPy dizisine çevir ki maskeleme çalışsın (train_labels aslında bir list)
    labels_np = np.array(labels)

    # Maske ile listeleri doğrudan böl
    labels_if_present = labels_np[present]
    labels_if_absent = labels_np[absent]

    # P(X_t=1) ve P(X_t=0) hesapla (doğrudan present.mean() de diyebilirsin)
    p1 = present.mean()
    p0 = absent.mean()

    # Calculate conditional entropies H(C | X_t=1) and H(C | X_t=0)
    H1 = entropy(labels_if_present) if p1 > 0 else 0.0
    H0 = entropy(labels_if_absent) if p0 > 0 else 0.0

    return Hc - (p1 * H1 + p0 * H0)

# Calculate H(C)
H_class = entropy(train_labels)
print(f"Entropy of the class variable H(C): {H_class:.4f} bits\n")

# Calculate IG for each word
IG = {w: info_gain(i, X_bin, train_labels) for w, i in word_to_idx.items()}

# Rank terms by IG in descending order
ig_ranked_terms = sorted(IG.items(), key=lambda item: item[1], reverse=True)

print("Information Gain (IG) for each term (ranked descending):")
ig_df = pd.DataFrame(ig_ranked_terms, columns=['Term', 'Information Gain (bits)'])
display(ig_df)


Entropy of the class variable H(C): 0.9710 bits

Information Gain (IG) for each term (ranked descending):


,Term,Information Gain (bits)
0,discount,0.970951
1,free,0.970951
2,pleasure,0.970951
3,cat,0.419973
4,mouse,0.419973
5,buy,0.321928
6,combo,0.321928
7,smile,0.321928
8,dog,0.170951


## Task 3: Mutual Information (MI) Calculation

For every term $t$, we compute the (pointwise-summed) Mutual Information between the binary feature $X_t$ and the class $C$: $MI(X_t ; C) = \Sigma_{x\in\{0,1\}} \Sigma_{c\in\{S,N\}} P(x, c) \cdot \log_2 ( P(x, c) / ( P(x) P(c) ) )$.

We will use add-one (Laplace) smoothing on the joint counts to avoid $\log(0)$ and then rank all terms by MI in descending order. Finally, we will compare the IG and MI rankings and explain any differences for binary features.

In [96]:
# 5. Mutual Information (with add-one smoothing) -----------------------
def mutual_info(term_idx, X_bin, labels, alpha=1.0):
    classes = sorted(list(set(labels)))
    mi = 0.0
    n_docs = len(labels)

    # Ortak uzay smoothing paydası: N + |X| * |C| * alpha (burada |X|=2, |C|=2)
    # N_total = n_docs + (2 * len(classes) * alpha)
    n_smoothed_total = n_docs + (2 * len(classes) * alpha)

    labels_np = np.array(labels)
    present = X_bin[:, term_idx] == 1
    absent = ~present

    # x=1 (present) ve x=0 (absent) durumları için döngü
    for x_val, docs_mask in [(1, present), (0, absent)]:

        # Smoothed marginal P(x) = (count(x) + |C|*alpha) / N_total
        # |C| ile çarpıyoruz çünkü x, 2 farklı sınıf üzerinden toplanıyor.
        n_x = docs_mask.sum() + len(classes) * alpha
        p_x = n_x / n_smoothed_total

        for c in classes:
            # Smoothed marginal P(c) = (count(c) + |X|*alpha) / N_total
            # |X| ile çarpıyoruz çünkü c, x'in 2 farklı durumu (var/yok) üzerinden toplanıyor.
            n_c = (labels_np == c).sum() + 2 * alpha # 2 for |X| (present/absent)
            p_c = n_c / n_smoothed_total

            # Smoothed joint P(x, c) = (count(x, c) + alpha) / N_total
            n_xc = (docs_mask & (labels_np == c)).sum() + alpha
            p_xc = n_xc / n_smoothed_total

            if p_xc > 0 and p_x > 0 and p_c > 0:
                mi += p_xc * np.log2(p_xc / (p_x * p_c))

    return mi

MI = {w: mutual_info(i, X_bin, train_labels) for w, i in word_to_idx.items()}

# Rank terms by MI in descending order
mi_ranked_terms = sorted(MI.items(), key=lambda item: item[1], reverse=True)

print("Mutual Information (MI) for each term (ranked descending, with add-one smoothing):")
mi_df = pd.DataFrame(mi_ranked_terms, columns=['Term', 'Mutual Information (bits)'])
display(mi_df)


Mutual Information (MI) for each term (ranked descending, with add-one smoothing):


,Term,Mutual Information (bits)
0,discount,0.229437
1,free,0.229437
2,pleasure,0.229437
3,cat,0.091091
4,mouse,0.091091
5,buy,0.072780
6,combo,0.072780
7,smile,0.072780
8,dog,0.018311


### Comparison of IG and MI Rankings

Let's compare the rankings obtained from Information Gain (IG) and Mutual Information (MI):

**Information Gain Ranking:**


In [102]:
print("Information Gain (IG) for each term (ranked descending, with add-one smoothing):")
display(ig_df)

Information Gain (IG) for each term (ranked descending, with add-one smoothing):


,Term,Information Gain (bits)
0,discount,0.970951
1,free,0.970951
2,pleasure,0.970951
3,cat,0.419973
4,mouse,0.419973
5,buy,0.321928
6,combo,0.321928
7,smile,0.321928
8,dog,0.170951



**Mutual Information Ranking:**


In [98]:
display(mi_df)

,Term,Mutual Information (bits)
0,discount,0.229437
1,free,0.229437
2,pleasure,0.229437
3,cat,0.091091
4,mouse,0.091091
5,buy,0.072780
6,combo,0.072780
7,smile,0.072780
8,dog,0.018311


For binary features, Information Gain (IG) and Mutual Information (MI) are mathematically equivalent. The core difference between IG and MI in their standard definitions is that IG uses conditional entropy $H(C|X_t)$ and subtracts it from the overall entropy $H(C)$, while MI directly measures the reduction in uncertainty of $C$ given $X_t$ (or vice-versa) by summing over joint probabilities.

Specifically, for binary features, it can be shown that $IG(X_t) = H(C) - H(C|X_t)$ is indeed equal to $MI(X_t; C) = \sum_{x \in \{0,1\}} \sum_{c \in \{S,N\}} P(x,c) \log_2 \frac{P(x,c)}{P(x)P(c)}$. Therefore, their rankings should be identical, provided the same probability estimation and smoothing techniques are used.

Any slight numerical differences observed in practice might stem from floating-point precision or very minor variations in the implementation of the probability estimations, especially with smoothing. However, conceptually and mathematically for binary features, they quantify the same measure of dependence between the feature and the class.

## Task 4: Feature Selection

In this task, we will select the top-k most informative terms based on their Information Gain (IG) scores. We will consider `k = 2`, `k = 3`, and `k = 5`. For each selected set of features, we will then build reduced feature matrices for both the training documents and the test documents (d6 and d7).

In [99]:
# 6. Feature selection (top-k by IG) -----------------------------------
def top_k_features(scores, k):
    # IG skorlarına göre azalan (descending) sırada k tane kelimeyi seç
    # reverse=True kullanmak -kv[1]'den daha okunaklı bir standarttır.
    return [w for w, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:k]]

k_values = [2, 3, 5]
reduced_train_matrices = {}
reduced_test_matrices = {}
selected_features_sets = {}

for k in k_values:
    print(f"\n--- Processing for k = {k} features ---")

    # 1. Top-K özellikleri seç
    selected_features = top_k_features(IG, k)
    selected_features_sets[k] = selected_features
    print(f"Top {k} features: {selected_features}")

    # 2. Seçilen kelimelerin Task 1'deki orijinal matristeki kolon indekslerini bul
    selected_indices = [word_to_idx[w] for w in selected_features]

    # 3. EĞİTİM MATRİSİ: Yeniden kelime saymak yerine X_bin'i direkt o kolonlardan kes! (Çok Hızlı)
    X_train_reduced_bin = X_bin[:, selected_indices]
    reduced_train_matrices[k] = X_train_reduced_bin

    df_X_train_reduced = pd.DataFrame(X_train_reduced_bin, columns=selected_features, index=[f'd{i+1}' for i in range(len(train_docs))])
    print(f"Reduced Training Binary Matrix (k={k}):")
    display(df_X_train_reduced)

    # 4. TEST MATRİSİ: Test dokümanları için (k boyutu kadar) sıfır matrisi oluştur ve sadece seçili kelimeleri ara
    X_test_reduced_bin = np.zeros((len(test_docs), k), dtype=int)
    for i, doc in enumerate(test_docs):
        for j, w in enumerate(selected_features):
            if w in doc:
                X_test_reduced_bin[i, j] = 1

    reduced_test_matrices[k] = X_test_reduced_bin

    df_X_test_reduced = pd.DataFrame(X_test_reduced_bin, columns=selected_features, index=[f'd{i+6}' for i in range(len(test_docs))])
    print(f"Reduced Test Binary Matrix (k={k}):")
    display(df_X_test_reduced)



--- Processing for k = 2 features ---
Top 2 features: ['discount', 'free']
Reduced Training Binary Matrix (k=2):


,discount,free
d1,1,1
d2,1,1
d3,0,0
d4,0,0
d5,0,0


Reduced Test Binary Matrix (k=2):


,discount,free
d6,0,0
d7,0,1



--- Processing for k = 3 features ---
Top 3 features: ['discount', 'free', 'pleasure']
Reduced Training Binary Matrix (k=3):


...

,discount,free,pleasure
d6,0,0,0
d7,0,1,0



--- Processing for k = 5 features ---
Top 5 features: ['discount', 'free', 'pleasure', 'cat', 'mouse']
Reduced Training Binary Matrix (k=5):


,discount,free,pleasure,cat,mouse
d1,1,1,1,0,0
d2,1,1,1,0,0
d3,0,0,0,1,1
d4,0,0,0,1,0
d5,0,0,0,0,1


Reduced Test Binary Matrix (k=5):


,discount,free,pleasure,cat,mouse
d6,0,0,0,1,1
d7,0,1,0,0,0


## Task 5: kNN classification

In this task, we will implement a simple kNN classifier from scratch, using cosine similarity as the distance metric. We will then use this classifier to predict the class of the test documents (d6 and d7) for different combinations of `k` (number of neighbors) and the feature-set sizes selected in Task 4. Ties will be broken by majority vote, then by smaller distance.

In [100]:
def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    # Handle cases where one or both vectors are zero vectors
    return 0.0 if nu == 0 or nv == 0 else float(np.dot(u, v) / (nu*nv))

def knn_predict(X_tr, y_tr, x_te, k):
    sims = [cosine(x_te, x) for x in X_tr]

    if k == 0 or len(X_tr) == 0: # If no training data or k=0, cannot predict
        return None, []

    # Get indices of the top k neighbors, sorted by similarity in descending order
    # Use stable sort to preserve order for ties (important for tie-breaking)
    sorted_top_k_indices = np.argsort(sims, kind='stable')[::-1][:k]

    # Build list of top k neighbors with their info
    top_k_neighbors_info = []
    for idx in sorted_top_k_indices:
        top_k_neighbors_info.append({'index': idx, 'similarity': sims[idx], 'label': y_tr[idx]})

    # Perform majority voting on these top k neighbors
    votes = Counter(n['label'] for n in top_k_neighbors_info)

    max_votes = 0
    winning_classes = []
    for class_label, count in votes.items():
        if count > max_votes:
            max_votes = count
            winning_classes = [class_label]
        elif count == max_votes:
            winning_classes.append(class_label)

    predicted_class = None
    if len(winning_classes) == 1:
        predicted_class = winning_classes[0]
    else:
        # Tie-breaking for equal votes: choose the class whose neighbors
        # within the *top k* have a smaller distance (higher average similarity).
        best_avg_sim = -1.0
        # Iterate through winning_classes in a deterministic order (e.g., sorted)
        # to ensure consistent tie-breaking if average similarities are also equal
        for c in sorted(winning_classes):
            class_sims = [n['similarity'] for n in top_k_neighbors_info if n['label'] == c]
            avg_sim = sum(class_sims) / len(class_sims) if class_sims else 0.0

            if avg_sim > best_avg_sim:
                best_avg_sim = avg_sim
                predicted_class = c
            # If avg_sim == best_avg_sim, the 'sorted(winning_classes)' part
            # already provides a deterministic tie-break based on class name.

    # For display, strictly show only the top k neighbors as per user's latest feedback
    display_neighbors = [(n['index'], n['similarity'], n['label']) for n in top_k_neighbors_info]

    return predicted_class, display_neighbors

In [101]:
# 8. Run experiments for k in {1,3} and feature-set sizes {2,3,5}
#    Print neighbours, similarities, and predicted classes for d6 and d7.

k_values_knn = [1, 3]

print("\n--- kNN Classification Results ---")

# Prepare train_labels as a list to align with current knn_predict implementation
y_tr = train_labels

for feature_k in k_values: # Feature set sizes from Task 4: 2, 3, 5
    print(f"\nFeature Set Size: {feature_k} (Features: {selected_features_sets[feature_k]})")

    X_tr_reduced = reduced_train_matrices[feature_k]
    X_te_reduced = reduced_test_matrices[feature_k]

    for knn_k in k_values_knn: # k for kNN: 1, 3
        print(f"  kNN k = {knn_k}:")

        # Predict for d6
        print(f"    Test Document d6:")
        predicted_class_d6, neighbors_d6 = knn_predict(X_tr_reduced, y_tr, X_te_reduced[0], knn_k)
        print(f"      Predicted Class: {predicted_class_d6}")
        print(f"      Neighbors (index, similarity, label): {neighbors_d6}")

        # Predict for d7
        print(f"    Test Document d7:")
        predicted_class_d7, neighbors_d7 = knn_predict(X_tr_reduced, y_tr, X_te_reduced[1], knn_k)
        print(f"      Predicted Class: {predicted_class_d7}")
        print(f"      Neighbors (index, similarity, label): {neighbors_d7}")


--- kNN Classification Results ---

Feature Set Size: 2 (Features: ['discount', 'free'])
  kNN k = 1:
    Test Document d6:
      Predicted Class: N
      Neighbors (index, similarity, label): [(np.int64(4), 0.0, 'N')]
    Test Document d7:
      Predicted Class: S
      Neighbors (index, similarity, label): [(np.int64(1), 0.7071067811865475, 'S')]
  kNN k = 3:
    Test Document d6:
      Predicted Class: N
      Neighbors (index, similarity, label): [(np.int64(4), 0.0, 'N'), (np.int64(3), 0.0, 'N'), (np.int64(2), 0.0, 'N')]
    Test Document d7:
      Predicted Class: S
      Neighbors (index, similarity, label): [(np.int64(1), 0.7071067811865475, 'S'), (np.int64(0), 0.7071067811865475, 'S'), (np.int64(4), 0.0, 'N')]

Feature Set Size: 3 (Features: ['discount', 'free', 'pleasure'])
  kNN k = 1:
    Test Document d6:
      Predicted Class: N
      Neighbors (index, similarity, label): [(np.int64(4), 0.0, 'N')]
    Test Document d7:
      Predicted Class: S
      Neighbors (index, simi